## This notebook is specifically to filter the MET Dataset 

Formatting of the provenance record goes as follows (person/organization of ownership), location, date 

In [1]:
import pandas as pd # this imports the pandas library under the nickname 'pd' which is lused a lot of online documentation
import re
df = pd.read_json('met_scraped_provenance_data.json')
df = df.drop(columns="html")
df.head()
# df.loc[df['object_id']=='310325']

,page_name,object_id,provenance
0,Aduma or Mbete-Kota artist - Mvudi (mask) - Ad...,310842,
1,Akan artist - Bird claw gold weight - Asante -...,312390,"Sir Cecil Armitage, Ghana, collected after 189..."
2,Akan artist - Grasshopper gold weight - Akan p...,310871,"Mr. and Mrs. William W. Brill, New York, by 19..."
3,Akan artist - Shells gold weight - Akan people...,310913,"Mr. and Mrs. William W. Brill, New York, by 19..."
4,Anyi-Akan artist - Hornblower attendant figure...,310829,"[Dr. Marcel Lheureux, Paris, acquired 1920s]; ..."


In [2]:
seated_couple = df.loc[df['object_id']==310325] # finding ONE instance of the object that has ambiguous provenance
seated_couple

,page_name,object_id,provenance
208,_Barnes Foundation Master_ - Seated couple - D...,310325,"[Henri Kamer, Paris and New York]; Lester Wund..."


In [3]:
# shows the structure of the provenance data once split by semicolon
provenance_data = seated_couple.provenance.values[0].split(";") # this pulls the provenance data as a list of strings, separated at the semicolon
provenance_data

['[Henri Kamer, Paris and New York]', ' Lester Wunderman, New York, by 1977']

In [4]:
# this cell tests out the ability to extract provenance data from the scraped dataset

provenance_dictionaries = []
provenance_record_complete = False

for provenance_entry in provenance_data:
    if "[" in provenance_entry or "until" in provenance_entry:
        provenance_dict = {
            "complete": False,
            "data": provenance_entry
        }
        provenance_dictionaries.append(provenance_dict)
    elif len(provenance_entry.split(",")) != 3:
        provenance_dict = {
            "complete": False,
            "data": provenance_entry
        }
    else:
        provenance_fields = provenance_entry.split(",")
        name = provenance_fields[0]
        place = provenance_fields[1]
        date_matches = re.findall(r"\b\d{4}\b", provenance_fields[2])
        if(len(date_matches)) == 1:
            date = date_matches[0]
            complete = True
            provenance_record_complete = True
        else:
            complete = False
        provenance_dict = {
            "complete": complete,
            "name": name,
            "place": place,
            "date": date,
            "data": provenance_entry
        }
        provenance_dictionaries.append(provenance_dict)

provenance_dictionaries

[{'complete': False, 'data': '[Henri Kamer, Paris and New York]'},
 {'complete': True,
  'name': ' Lester Wunderman',
  'place': ' New York',
  'date': '1977',
  'data': ' Lester Wunderman, New York, by 1977'}]

In [5]:
# creating a function to extract provenance records

def extract_provenance_dictionary(record):
    val = record['provenance']
    if isinstance(val, pd.Series):
        val = val.values[0] if not val.empty else ""

    if isinstance(val,str) and val.strip():
        provenance_data = val.split(";") # this pulls the provenance data as a list of strings, separated at the semicolon
    else:
        return False, None
    
    if(len(provenance_data)==0):
        return False, None # returns False for "complete", and None for provenance record

    provenance_dictionaries = []
    provenance_record_complete = False
    
    for provenance_entry in provenance_data:
        # testing to see if a range of years is included
        pattern = r"^\d{4}-\d{4}$"
        rangematch = bool(re.match(pattern, provenance_entry))
        if "probably" in provenance_entry or "until" in provenance_entry or "by" in provenance_entry or rangematch == True:
            provenance_dict = {
                "complete": False,
                "data": provenance_entry
            }
            provenance_dictionaries.append(provenance_dict)
        elif len(provenance_entry.split(",")) != 3:
            provenance_dict = {
                "complete": False,
                "data": provenance_entry
            }
        else:
            provenance_fields = provenance_entry.split(",")
            name = provenance_fields[0]
            place = provenance_fields[1]
            date_matches = re.findall(r"\b\d{4}\b", provenance_fields[2])
            if(len(date_matches)) == 1:
                date = date_matches[0]
                complete = True
                provenance_record_complete = True
            else:
                complete = False
                provenance_record_complete = False
                date = None
            provenance_dict = {
            "complete": complete,
            "name": name,
            "place": place,
            "date": date,
            "data": provenance_entry
        }
        provenance_dictionaries.append(provenance_dict)

    for item in provenance_dictionaries:
        if item.get("complete") is not True:
            provenance_record_complete = False
            break

    return provenance_record_complete, provenance_dictionaries



In [6]:
objet = df.loc[df['object_id']==312390] # finding ONE instance of the object that has ambiguous provenance
objet

is_complete, data = extract_provenance_dictionary(objet)

is_complete


False

In [7]:
df[['provenance_complete','provenance_details']] = df.apply(extract_provenance_dictionary, axis=1, result_type='expand')

In [8]:
df.head()

,page_name,object_id,provenance,provenance_complete,provenance_details
0,Aduma or Mbete-Kota artist - Mvudi (mask) - Ad...,310842,,False,None
1,Akan artist - Bird claw gold weight - Asante -...,312390,"Sir Cecil Armitage, Ghana, collected after 189...",False,"[{'complete': False, 'data': 'Sir Cecil Armita..."
2,Akan artist - Grasshopper gold weight - Akan p...,310871,"Mr. and Mrs. William W. Brill, New York, by 19...",False,"[{'complete': False, 'data': 'Mr. and Mrs. Wil..."
3,Akan artist - Shells gold weight - Akan people...,310913,"Mr. and Mrs. William W. Brill, New York, by 19...",False,"[{'complete': False, 'data': 'Mr. and Mrs. Wil..."
4,Anyi-Akan artist - Hornblower attendant figure...,310829,"[Dr. Marcel Lheureux, Paris, acquired 1920s]; ...",False,"[{'complete': False, 'name': '[Dr. Marcel Lheu..."


In [9]:
# df.to_json("met_data_with_provenance_details.json", orient="records", indent=4)

## Refined provenance parser

The first version above had a few problems:
- steps with more or fewer than 3 comma-separated fields were dropped even if they had useful data 
- `"by" in provenance_entry` caught words that contain "by", like "Kirby"
- the date-range check only matched a step that was *nothing but* a range, and only with a hyphen, while the Met uses an en dash (`1959–78`)
- bracketed steps (dealers) were thrown away instead of parsed

The version below parses every step, keeps the dealer information in a `bracketed` key, records numeric start/end years so we can measure gaps, and stores the reasons a step is considered incomplete.

In [10]:
# had to do some research on regular expressions here to catch not only single years but ranges of years
# which appear in a number of ways in the dataset. These are the regular expression patterns that will be used
# to find specific matches in the provenance records

import re
import pandas as pd

# A four-digit year between 1000 and 2099.
# The parentheses make it a "capture group" so we can pull the year out later.
YEAR = r"(1[0-9]{3}|20[0-9]{2})"

# A year that may be followed by an end year, e.g.:
#   1959          -> a single year
#   1959–78       -> a range with a 2-digit end year (en dash, as the Met writes it)
#   1959-1978     -> a range with a 4-digit end year (normal hyphen)
#   1920s         -> a decade (the optional "s")
#   1897(?)       -> an uncertain year (the optional "(?)")
#   1940–(d.) 1950 -> a range ending at the owner's death
# Piece by piece:
#   YEAR                        the start year (capture group 1)
#   s?                          optional "s" for decades
#   (?:\s*\(\?\))?              optional "(?)"
#   (?:\s*[–-]\s*               optional dash (en dash OR hyphen) with spaces around it...
#      (?:\(d\.\)\s*)?          ...optionally followed by "(d.)"...
#      (\d{4}|\d{2})\b)?        ...and then the end year, 4 or 2 digits (capture group 2)
YEAR_OR_RANGE = re.compile(YEAR + r"s?(?:\s*\(\?\))?(?:\s*[–-]\s*(?:\(d\.\)\s*)?(\d{4}|\d{2})\b)?")

# Words that tell us a step describes an auction sale rather than an owner.
# \b means "word boundary", so "sale" won't match those characters inside words (like wholesale)
# "no. 334" is the lot number, which only appears for auctions.
# re.IGNORECASE makes "Sotheby's" and "sotheby's" both match.
AUCTION = re.compile(r"sotheby|christie|drouot|bonhams|parke-bernet|\bsale\b|\bno\. ?\d+", re.IGNORECASE)

# Words and symbols that make a date uncertain ("fuzzy").
# Again \b ... \b makes sure we match whole words only, so "by" will NOT
# match inside a name like "Kirby" (a bug in the first version).
#   \?      a literal question mark, e.g. "1897(?)" or "Paris (?)"
#   \d{4}s  a decade like "1920s"
FUZZY_WORDS = re.compile(
    r"\b(by|until|probably|possibly|ca\.|circa|before|after|at least|about|early|late|mid)\b|\?|\d{4}s",
    re.IGNORECASE,
)

In [11]:
# parse_dates: turn the date text of a step into two numbers rather than a string

def parse_dates(text):
    # finditer returns every place in the text where the pattern matches
    matches = list(YEAR_OR_RANGE.finditer(text))

    # no year at all -> we can't date this step, return None, None
    if not matches:
        return None, None

    # the START is the first year mentioned (matches[0])
    start = int(matches[0].group(1))

    # the END is the last year mentioned: either the end of a range
    # (group 2) or, if there's no range, the year itself (group 1)
    last = matches[-1] # index [-1] is the last one
    end = last.group(2) or last.group(1)

    # the Met often shortens the end of a range: "1959–78" means 1978.
    # If the end year only has 2 digits, borrow the first 2 digits ("19")
    # from the start of that range. Tricky!
    if len(end) == 2:
        end = last.group(1)[:2] + end

    return start, int(end)


# quick checks so you can see what the function does
print(parse_dates("by 1977"))
print(parse_dates("1959–78"))
print(parse_dates("by 1935, until 1957"))
print(parse_dates("Paris"))

(1977, 1977)
(1959, 1978)
(1935, 1957)
(None, None)


In [12]:
# parse_step: turn ONE provenance step (the text between two semicolons)
#             into a dictionary

def parse_step(raw, position):
    """
    raw      -> the text of one step, e.g. "[John J. Klejman, New York, by 1959]"
    position -> where this step falls in the timeline (0 = earliest)

    Returns a dictionary with name, place, dates, flags and the original text.
    """
    # Remove surrounding spaces and a trailing period ("...1955–1978.")
    text = raw.strip().rstrip(".").strip()

    # BRACKETS AND PARENTHESES 
    # The Met puts DEALERS in square brackets: [Henri Kamer, Paris]
    bracketed = text.startswith("[") and text.endswith("]")
    # Steps fully in parentheses are auction sales:
    # (Christie's, London, April 3, 1981, no. 334)
    in_parens = text.startswith("(") and text.endswith(")")
    # now that we have determined if brackets/parentheses are there, and we've stored it...
    # Strip the outer brackets/parentheses so the rest of the
    # parsing works the same way for every step.
    # text[1:-1] means "everything except the first and last character".
    if bracketed or in_parens:
        text = text[1:-1].strip()

    # Some auctions are in brackets instead of parentheses, so we
    # also look for auction-house names and lot numbers. These are edge cases, but worth catching here!
    auction = in_parens or bool(AUCTION.search(text))

    # "on loan to ..."
    # e.g. "Nelson A. Rockefeller, New York, 1960, on loan to The Museum of
    # Primitive Art, New York, 1960–1978"
    # The part after "on loan to" describes a DIFFERENT institution, so we
    # cut it off and store it separately. Otherwise its place and dates
    # would get mixed up with Rockefeller's.
    # \s? allows for the typo "onloan" that appears in the data.
    loan = None
    loan_match = re.search(r",?\s*on\s?loan to\b(.*)$", text, re.IGNORECASE)
    if loan_match:
        loan = loan_match.group(1).strip(" ,")          # what comes after "on loan to"
        text = text[:loan_match.start()].strip(" ,")    # what comes before it

    # SPLIT INTO COMMA-SEPARATED FIELDS for name place and date
    # The Met's pattern is roughly: name, place(s), date
    # but the number of fields varies, e.g.
    #   "Lester Wunderman, New York, by 1977"            -> 3 fields
    #   "Augustus Lane-Fox Pitt-Rivers, Farnham, Dorset, U.K., 1899–1958" -> 5 fields
    #   "Donald H. Peters"                               -> 1 field
    # So instead of demanding exactly 3 fields, we:
    #   - take the FIRST field as the name
    #   - treat any later field containing a digit as date information
    #   - treat any later field without a digit as place information
    fields = [f.strip() for f in text.split(",") if f.strip()]
    name = fields[0] if fields else None
    rest = fields[1:]

    place = ", ".join(f for f in rest if not re.search(r"\d", f)) or None
    date_text = ", ".join(f for f in rest if re.search(r"\d", f)) or None
    # ("or None" turns an empty string "" into None, so missing values
    #  show up as NaN in pandas instead of a blank string)

    # DATES
    # Use the owner's own date if there is one. If not, fall back on the
    # dates of the loan, since that's the only time information we have.
    date_start, date_end = parse_dates(date_text or "")
    if date_start is None and loan:
        date_start, date_end = parse_dates(loan)

    # DETERMINE IF THE PROVENANCE STEP RECORD IS COMPLETE
    # Instead of just True/False, we keep a LIST of reasons, so later we can
    # count which kinds of uncertainty are most common, or change our mind
    # about what counts as "fuzzy" without re-parsing everything.
    fuzzy_reasons = []
    if date_start is None:
        fuzzy_reasons.append("no date")
    elif date_start != date_end:
        # NOTE: a range like "1959–78" has a known start AND end, so you may
        # decide it shouldn't count as fuzzy. Delete these two lines if so.
        fuzzy_reasons.append("range")

    # RECORD WHY WE THOUGHT IT WAS INCOMPLETE
    for m in FUZZY_WORDS.finditer(raw):
        # group(1) is the matched word (e.g. "by"); for "?" or "1920s"
        # there's no group(1), so we use the whole match, group(0)
        fuzzy_reasons.append(m.group(1).lower() if m.group(1) else m.group(0))

    # REMOVE DUPLICATES
    fuzzy_reasons = list(dict.fromkeys(fuzzy_reasons))

    # --- 6. Build the dictionary --------------------------------------------
    return {
        "position": position,            # order in the timeline (0 = first)
        "name": name,                    # collector, dealer or institution
        "place": place,                  # city / country, may have several
        "date_text": date_text,          # the date as written, e.g. "by 1959"
        "date_start": date_start,        # first year as a number
        "date_end": date_end,            # last year as a number
        "bracketed": bracketed,          # True = dealer
        "auction": auction,              # True = auction sale
        "on_loan_to": loan,              # institution it was lent to, if any
        "complete": len(fuzzy_reasons) == 0,  # True only if nothing is uncertain
        "fuzzy_reasons": fuzzy_reasons,  # why it's incomplete
        "raw": raw.strip(),              # the original text, for checking
    }


# try it on one step
parse_step("[John J. Klejman, New York, by 1959]", 0)

{'position': 0,
 'name': 'John J. Klejman',
 'place': 'New York',
 'date_text': 'by 1959',
 'date_start': 1959,
 'date_end': 1959,
 'bracketed': True,
 'auction': False,
 'on_loan_to': None,
 'complete': False,
 'fuzzy_reasons': ['by'],
 'raw': '[John J. Klejman, New York, by 1959]'}

In [13]:
# extract_provenance_dictionary: parse the WHOLE provenance record of one object
def extract_provenance_dictionary(record):
    """
    record -> one row of the dataframe (df.apply passes each row in turn)

    Returns two values:
      1. True if EVERY step in the record is complete, otherwise False
      2. a list of step dictionaries (from parse_step), in timeline order
    """
    val = record["provenance"]

    # Some objects have an empty provenance ("") and some have NaN, which
    # pandas stores as a float. isinstance(val, str) catches both cases.
    if not isinstance(val, str) or not val.strip():
        return False, []

    # Split the record into steps at each semicolon, skipping empty pieces
    # (e.g. from a ";" at the very end of the record)
    steps = [s for s in val.split(";") if s.strip()]

    # Parse every step. enumerate gives us the position (0, 1, 2, ...)
    # along with the text of the step.
    details = [parse_step(s, i) for i, s in enumerate(steps)]

    # all(...) is True only if every step has complete == True
    record_complete = all(d["complete"] for d in details)

    return record_complete, details


# test on the "seated couple" object from earlier
extract_provenance_dictionary(df.loc[df["object_id"] == 310325].iloc[0])

(False,
 [{'position': 0,
   'name': 'Henri Kamer',
   'place': 'Paris and New York',
   'date_text': None,
   'date_start': None,
   'date_end': None,
   'bracketed': True,
   'auction': False,
   'on_loan_to': None,
   'complete': False,
   'fuzzy_reasons': ['no date'],
   'raw': '[Henri Kamer, Paris and New York]'},
  {'position': 1,
   'name': 'Lester Wunderman',
   'place': 'New York',
   'date_text': 'by 1977',
   'date_start': 1977,
   'date_end': 1977,
   'bracketed': False,
   'auction': False,
   'on_loan_to': None,
   'complete': False,
   'fuzzy_reasons': ['by'],
   'raw': 'Lester Wunderman, New York, by 1977'}])

In [14]:
# APPLY PARSER TO EVERY RECORD
# axis=1 runs the function once per ROW.
# result_type="expand" splits the two returned values into two new columns.
df[["provenance_complete", "provenance_details"]] = df.apply(
    extract_provenance_dictionary, axis=1, result_type="expand"
)

print("objects:", len(df))
print("objects with a fully complete record:", df["provenance_complete"].sum())
df[["object_id", "provenance_complete", "provenance_details"]].head()

objects: 210
objects with a fully complete record: 3


,object_id,provenance_complete,provenance_details
0,310842,False,[]
1,312390,False,"[{'position': 0, 'name': 'Sir Cecil Armitage',..."
2,310871,False,"[{'position': 0, 'name': 'Mr. and Mrs. William..."
3,310913,False,"[{'position': 0, 'name': 'Mr. and Mrs. William..."
4,310829,False,"[{'position': 0, 'name': 'Dr. Marcel Lheureux'..."


### One row per provenance step

`provenance_details` holds a *list* of dictionaries per object, which is hard to count or filter. Below we "explode" it so every provenance step becomes its own row, with one column per dictionary key. This table is what we use for name counts and gap analysis.

In [15]:
# explode: one row per item in each list (one row per step).
# dropna removes objects that had no provenance at all.
steps = df[["object_id", "provenance_details"]].explode("provenance_details").dropna()

# json_normalize turns each dictionary into columns (name, place, date_start, ...)
# We reset the index on both sides so the rows line up when we glue them together.
steps = pd.concat(
    [
        steps[["object_id"]].reset_index(drop=True),
        pd.json_normalize(steps["provenance_details"].tolist()),
    ],
    axis=1,
)

print("total steps:", len(steps))
steps.head(10)

total steps: 525


,object_id,position,name,place,date_text,date_start,date_end,bracketed,auction,on_loan_to,complete,fuzzy_reasons,raw
0,312390,0,Sir Cecil Armitage,Ghana,"collected after 1898, until (d.)1933",1898.0,1933.0,False,False,None,False,"[range, after, until]","Sir Cecil Armitage, Ghana, collected after 189..."
1,312390,1,Maurice S. Cockin,London,1933–(d.)1960,1933.0,1960.0,False,False,None,False,[range],"Maurice S. Cockin, London, 1933–(d.)1960"
2,312390,2,Nelson A. Rockefeller,New York,1960,1960.0,1960.0,False,False,"The Museum of Primitive Art, New York, 1960–1978",True,[],"Nelson A. Rockefeller, New York, 1960, on loan..."
3,310871,0,Mr. and Mrs. William W. Brill,New York,by 1961,1961.0,1961.0,False,False,None,False,[by],"Mr. and Mrs. William W. Brill, New York, by 1961"
4,310871,1,The Museum of Primitive Art,New York,1961–1978,1961.0,1978.0,False,False,None,False,[range],"The Museum of Primitive Art, New York, 1961–1978"
5,310913,0,Mr. and Mrs. William W. Brill,New York,by 1961,1961.0,1961.0,False,False,None,False,[by],"Mr. and Mrs. William W. Brill, New York, by 1961"
6,310913,1,The Museum of Primitive Art,New York,1961–1978,1961.0,1978.0,False,False,None,False,[range],"The Museum of Primitive Art, New York, 1961–1978"
7,310829,0,Dr. Marcel Lheureux,Paris,acquired 1920s,1920.0,1920.0,True,False,None,False,[1920s],"[Dr. Marcel Lheureux, Paris, acquired 1920s]"
8,310829,1,P. Alsace,None,None,NaN,NaN,False,False,None,False,[no date],P. Alsace
9,310829,2,John J. Klejman,New York,by 1960,1960.0,1960.0,True,False,None,False,[by],"[John J. Klejman, New York, by 1960]"


In [23]:
# Which kinds of uncertainty are most common?
# fuzzy_reasons is a list per step, so we explode it again to count each reason
steps["fuzzy_reasons"].explode().value_counts() # for bar chart version .plot(kind = 'barh')

fuzzy_reasons
by          195
range       172
no date      45
until        28
?             8
possibly      7
before        5
after         2
1950s         2
1920s         2
1930s         1
probably      1
1960s         1
ca.           1
at least      1
1970s         1
early         1
Name: count, dtype: int64

In [ ]:
# ----------------------------------------------------------------------------
# Names that appear most often (all steps)
# ----------------------------------------------------------------------------
steps["name"].value_counts().head(20)

In [ ]:
# ----------------------------------------------------------------------------
# Names that appear most often among DEALERS (bracketed steps)
# ----------------------------------------------------------------------------
# & means "and"; ~ means "not". So this keeps bracketed steps that are
# not auction sales (some auctions are also written in brackets).
dealers = steps[steps["bracketed"] & ~steps["auction"]]
dealers["name"].value_counts().head(20)

In [ ]:
# Gaps between owners
# Sort so each object's steps are in timeline order.
steps = steps.sort_values(["object_id", "position"])

# For each step, look at the NEXT step of the same object (shift(-1) moves
# the column up by one row within each object_id group) and subtract this
# owner's end year from the next owner's start year.
#   positive number -> years with no recorded owner
#   NaN             -> one of the two steps has no date
steps["gap_years"] = (
    steps.groupby("object_id")["date_start"].shift(-1) - steps["date_end"]
)

# The 10 largest gaps
steps.sort_values("gap_years", ascending=False)[
    ["object_id", "position", "name", "date_end", "gap_years", "raw"]
].head(10)

In [ ]:
# Save the one-row-per-step table so it can be reused without re-parsing
steps.to_json("met_provenance_steps.json", orient="records", indent=4)

### Gap between an object's date of origin and its first provenance record

The scraped pages only give us the provenance text. The object's estimated date of creation comes from the Met API data (`met_api_object_data.json`), which has two numeric fields:
- `objectBeginDate`: the earliest year the object could have been made
- `objectEndDate`: the latest year the object could have been made

For example, "19th–early 20th century" becomes 1800–1930. Because the date of origin is itself a range, we calculate two gaps:
- **`origin_gap_min`** = first provenance year − `objectEndDate`, the *shortest* possible gap (if the object was made as late as possible)
- **`origin_gap_max`** = first provenance year − `objectBeginDate`, the *longest* possible gap (if it was made as early as possible)

In [ ]:
# ----------------------------------------------------------------------------
# Load the object dates from the Met API data
# ----------------------------------------------------------------------------
# We only need the ID and the date columns. objectDate is the date as the Met
# writes it (e.g. "ca. 1880"), which is handy for checking our numbers.
api = pd.read_json("met_api_object_data.json")
object_dates = api[["objectID", "objectDate", "objectBeginDate", "objectEndDate"]]

# Rename objectID so it matches the column name in our steps table
object_dates = object_dates.rename(columns={"objectID": "object_id"})
object_dates.head()

In [ ]:
# ----------------------------------------------------------------------------
# Find the FIRST DATED provenance step for each object
# ----------------------------------------------------------------------------
# The very first step (position 0) often has no date. For example, "Court of
# Benin" or "Private collection, Europe". So we keep only steps that have a
# date_start, then take the earliest position per object.
dated_steps = steps.dropna(subset=["date_start"])

# sort by position, then keep the first row for each object_id
first_dated = (
    dated_steps.sort_values(["object_id", "position"])
    .groupby("object_id")
    .first()
    .reset_index()
)

# Keep only the columns we need, renamed so it's clear they describe
# the first dated step
first_dated = first_dated[["object_id", "position", "name", "date_start", "raw"]].rename(
    columns={
        "position": "first_dated_position",
        "name": "first_dated_name",
        "date_start": "first_provenance_year",
        "raw": "first_dated_raw",
    }
)

# first_dated_position > 0 means there are EARLIER steps that have no date,
# so the real gap may be shorter than we can measure
first_dated["undated_steps_before"] = first_dated["first_dated_position"]
first_dated.head()

###### ----------------------------------------------------------------------------
# Join the two tables and calculate the gaps
# ----------------------------------------------------------------------------
# merge works like a spreadsheet lookup: it matches rows on object_id.
# how="left" keeps every object, even ones with no dated provenance
# (their gap columns will be NaN).
origin_gaps = object_dates.merge(first_dated, on="object_id", how="left")

# shortest possible gap: object made as LATE as possible
origin_gaps["origin_gap_min"] = origin_gaps["first_provenance_year"] - origin_gaps["objectEndDate"]
# longest possible gap: object made as EARLY as possible
origin_gaps["origin_gap_max"] = origin_gaps["first_provenance_year"] - origin_gaps["objectBeginDate"]

# A NEGATIVE origin_gap_min just means the first owner is recorded before the
# end of the estimated creation range (e.g. made "1800–1930", collected 1900).
# That isn't an error; the object may have been collected soon after it was made.
# clip(lower=0) turns those negative numbers into 0.
origin_gaps["origin_gap_min"] = origin_gaps["origin_gap_min"].clip(lower=0)

origin_gaps.head()

In [ ]:
# ----------------------------------------------------------------------------
# Objects with the largest gaps between origin and first provenance
# ----------------------------------------------------------------------------
# Sorted by the SHORTEST possible gap, because that's the gap we can be most
# confident about: even in the best case, this many years are unaccounted for.
origin_gaps.sort_values("origin_gap_min", ascending=False)[
    ["object_id", "objectDate", "first_provenance_year", "origin_gap_min",
     "origin_gap_max", "undated_steps_before", "first_dated_name"]
].head(15)

In [ ]:
# ----------------------------------------------------------------------------
# Summary statistics
# ----------------------------------------------------------------------------
# describe() gives count, mean, median (50%), min and max for each column
print("objects with no dated provenance at all:", origin_gaps["first_provenance_year"].isna().sum())
origin_gaps[["origin_gap_min", "origin_gap_max"]].describe()

Parsing the met data has had challenge because it doesnt follow an exact pattern my methodology will be to have it formatted into (name, place, year) but include perameters because sometimes there's no year or place or both city and country are present or two possible locations

In [ ]:
# checking to see how many complete entries there are

complete_entries = df[df['provenance_complete'] == True]
complete_entries

Object id in API data and object number are different [312427](https://www.metmuseum.org/art/collection/search/312427) , [312428](https://www.metmuseum.org/art/collection/search/312428) , [312380](https://www.metmuseum.org/art/collection/search/312380). The first two objects with complete provenance data are a set and will follow the same provenance path. 